# Step 6: publish the adapter and test the server

1. **Attach the adapter**: Add Input → Your Work → the training notebook (or the dataset made from `router_adapter.zip`).
2. **Hugging Face write token**: huggingface.co → Settings → Access Tokens → *Create new token*, type **Write**. In Kaggle: Add-ons → Secrets → `HF_TOKEN` (tick it for this notebook). A read-only token will fail at the upload.
3. **Settings → GPU T4 ×2, Internet On.** Run All (~10 min).

It creates `<your-hf-username>/minicpm5-2b-jev-router` on the Hub, uploads the adapter, `calibration.json` and the model card, then starts the server from the GitHub repo and asks it a few routing questions.

In [ ]:
HF_REPO_NAME = "minicpm5-2b-jev-router"
PRIVATE = False
GITHUB_REPO = "ypg199/minicpm5-jev-router"
TEST_SERVER = True

In [ ]:
!pip install -q -U "transformers>=5.6" peft accelerate huggingface_hub fastapi uvicorn
!pip uninstall -y -q torchao   # Kaggle's torchao 0.10 is too old for current peft
import sys
for _m in [m for m in sys.modules if m.split(".")[0] in ("transformers", "peft", "accelerate", "torchao", "huggingface_hub")]:
    del sys.modules[_m]

In [ ]:
import os, glob, json, shutil, subprocess, time, urllib.request

hits = sorted(glob.glob("/kaggle/input/**/adapter_config.json", recursive=True))
hits = [h for h in hits if "adapter_last" not in h] or hits
assert hits, "No adapter found: attach the training notebook's output or the adapter dataset (Add Input)"
ADAPTER_DIR = os.path.dirname(hits[0])
print("adapter:", ADAPTER_DIR, sorted(os.listdir(ADAPTER_DIR)))
calib = json.load(open(os.path.join(ADAPTER_DIR, "calibration.json")))
print({k: calib[k] for k in ("base_model", "temperature", "joiner", "val_accuracy", "val_ece_raw", "val_ece_calibrated")})

In [ ]:
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("HF_TOKEN")
api = HfApi(token=token)
user = api.whoami()["name"]
HF_REPO = f"{user}/{HF_REPO_NAME}"

card = urllib.request.urlopen(f"https://raw.githubusercontent.com/{GITHUB_REPO}/main/MODEL_CARD.md").read().decode()
card = card.replace("<GITHUB_REPO>", GITHUB_REPO).replace("<HF_REPO>", HF_REPO)

stage = "/kaggle/working/hub_upload"
shutil.rmtree(stage, ignore_errors=True)
os.makedirs(stage)
for f in os.listdir(ADAPTER_DIR):
    if f.startswith("adapter_") or f == "calibration.json":
        shutil.copy(os.path.join(ADAPTER_DIR, f), stage)
open(os.path.join(stage, "README.md"), "w").write(card)

api.create_repo(HF_REPO, private=PRIVATE, exist_ok=True)
api.upload_folder(folder_path=stage, repo_id=HF_REPO, commit_message="Upload LoRA adapter")
print("published: https://huggingface.co/" + HF_REPO)

In [ ]:
# Smoke-test the server from the GitHub repo against the local adapter.
if TEST_SERVER:
    if not os.path.exists("/kaggle/working/router-src"):
        subprocess.run(["git", "clone", "-q", f"https://github.com/{GITHUB_REPO}", "/kaggle/working/router-src"], check=True)
    env = dict(os.environ, ROUTER_ADAPTER=ADAPTER_DIR)
    log = open("/kaggle/working/server.log", "w")
    server = subprocess.Popen([sys.executable, "-m", "uvicorn", "router.server:app", "--port", "8009"],
                              cwd="/kaggle/working/router-src", env=env, stdout=log, stderr=subprocess.STDOUT)
    for _ in range(180):
        try:
            if json.loads(urllib.request.urlopen("http://127.0.0.1:8009/health").read())["ok"]:
                break
        except Exception:
            pass
        if server.poll() is not None:
            break
        time.sleep(5)
    else:
        print("server did not start in 15 minutes")
    if server.poll() is not None:
        print(open("/kaggle/working/server.log").read()[-3000:])
    else:
        out = subprocess.run([sys.executable, "examples/client.py"], cwd="/kaggle/working/router-src",
                             capture_output=True, text=True)
        print(out.stdout, out.stderr[-2000:])
        server.terminate()